In [38]:
"""
ERCOT DAM 电价预测 - 特征提取模块
==================================
功能: 将DAM小时级价格数据转换为预测特征数据集
输出: 每行代表一个预测任务 (在D-1日预测D日某小时的价格)

市场规则:
- D-1日 10:00 投标截止
- D-1日 06:00 预测需完成
- 预测提前量: 18-42小时

特征设计原则:
1. 所有特征在预测时点 (D-1日 06:00) 必须可用
2. 不能使用D日的任何实际数据
3. 最新可用的DAM价格是D-1日的HE24

作者: 李源
版本: 2.0
"""

import pandas as pd
import numpy as np
from typing import Dict, List, Optional, Tuple
from dataclasses import dataclass, field
import warnings
warnings.filterwarnings('ignore')


# =============================================================================
# 第一部分: 配置类
# =============================================================================

@dataclass
class DAMFeatureConfig:
    """
    DAM特征提取配置
    
    Attributes
    ----------
    hours : List[int]
        预测的小时列表 (1-24, 对应HE1-HE24)
    spike_threshold : float
        尖峰价格阈值 ($/MWh)
    peak_hours : List[int]
        峰时小时列表 (HE7-HE22)
    summer_months : List[int]
        夏季月份列表 (6-9月)
    min_history_days : int
        最少需要的历史天数 (用于计算滞后特征)
    """
    hours: List[int] = field(default_factory=lambda: list(range(1, 25)))
    spike_threshold: float = 100.0
    peak_hours: List[int] = field(default_factory=lambda: list(range(7, 23)))
    summer_months: List[int] = field(default_factory=lambda: [6, 7, 8, 9])
    min_history_days: int = 28  # 至少需要28天历史数据 (4周)


# =============================================================================
# 第二部分: 特征列表定义
# =============================================================================

# -----------------------------------------------------------------------------
# 完整特征列表 (35个特征)
# -----------------------------------------------------------------------------

ALL_FEATURES = [
    # === 目标时间特征 (10个) ===
    'hour',                         # 0  [CAT] 目标小时 (1-24, 即HE1-HE24)
    'day_of_week',                  # 1  [CAT] 目标日星期几 (0=周一, 6=周日)
    'day_of_month',                 # 2  [CAT] 目标日日期 (1-31)
    'month',                        # 3  [CAT] 目标日月份 (1-12)
    'quarter',                      # 4  [CAT] 目标日季度 (1-4)
    'week_of_year',                 # 5        目标日年内周数 (1-53)
    'is_weekend',                   # 6  [CAT] 目标日是否周末 (0/1)
    'is_holiday',                   # 7  [CAT] 目标日是否节假日 (0/1)
    'is_peak_hour',                 # 8  [CAT] 目标小时是否峰时 (0/1)
    'is_summer',                    # 9  [CAT] 目标日是否夏季 (0/1)
    
    # === 滞后特征 (12个) ===
    'dam_lag_24h',                  # 10       D-1日同小时价格 (最重要!)
    'dam_lag_24h_prev',             # 11       D-1日前一小时价格
    'dam_lag_24h_next',             # 12       D-1日后一小时价格
    'dam_lag_48h',                  # 13       D-2日同小时价格
    'dam_lag_168h',                 # 14       D-7日同小时价格 (上周同天)
    'dam_d1_mean',                  # 15       D-1日24小时均价
    'dam_d1_max',                   # 16       D-1日24小时最高价
    'dam_d1_min',                   # 17       D-1日24小时最低价
    'dam_d1_std',                   # 18       D-1日24小时标准差
    'dam_7d_mean',                  # 19       过去7天均价
    'dam_7d_same_hour_mean',        # 20       过去7天同小时均价
    'dam_4w_same_dow_hour_mean',    # 21       过去4周同星期同小时均价
    
    # === 模式特征 (8个) ===
    'dam_d1_hour_ratio',            # 22       D-1日该小时/D-1日均价
    'dam_7d_hour_ratio',            # 23       7天同小时均价/7天总均价
    'dam_d1_vs_d2',                 # 24       D-1日均价 - D-2日均价
    'dam_d1_vs_d7',                 # 25       D-1日均价 - D-7日均价
    'dam_trend_3d',                 # 26       3天价格趋势 (斜率)
    'dam_7d_cv',                    # 27       7天变异系数 (std/mean)
    'dam_d1_range',                 # 28       D-1日价格范围 (max-min)
    'dam_7d_same_hour_std',         # 29       7天同小时标准差
    
    # === 尖峰特征 (5个) ===
    'dam_d1_spike_count',           # 30       D-1日尖峰次数 (>阈值)
    'dam_7d_spike_count',           # 31       7天尖峰次数
    'dam_d1_had_spike',             # 32 [CAT] D-1日是否有尖峰 (0/1)
    'dam_lag_24h_is_spike',         # 33 [CAT] D-1日同小时是否尖峰 (0/1)
    'dam_d1_max_hour',              # 34       D-1日最高价出现的小时
]

# -----------------------------------------------------------------------------
# 分类特征索引
# -----------------------------------------------------------------------------

CATEGORICAL_INDICES = [0, 1, 2, 3, 4, 6, 7, 8, 9, 32, 33]

# 分类特征名称 (与索引对应)
CATEGORICAL_FEATURES = [
    'hour',              # 索引 0
    'day_of_week',       # 索引 1
    'day_of_month',      # 索引 2
    'month',             # 索引 3
    'quarter',           # 索引 4
    'is_weekend',        # 索引 6
    'is_holiday',        # 索引 7
    'is_peak_hour',      # 索引 8
    'is_summer',         # 索引 9
    'dam_d1_had_spike',  # 索引 32
    'dam_lag_24h_is_spike',  # 索引 33
]


# =============================================================================
# 第三部分: 节假日处理
# =============================================================================

# 美国主要节假日 (简化版)
US_MAJOR_HOLIDAYS = {
    # 固定日期节日
    (1, 1),    # 元旦 New Year's Day
    (7, 4),    # 独立日 Independence Day
    (11, 11),  # 退伍军人节 Veterans Day
    (12, 25),  # 圣诞节 Christmas Day
    (12, 31),  # 新年前夜 New Year's Eve
}

def is_us_holiday(dt: pd.Timestamp) -> int:
    """
    判断是否为美国节假日 (简化版)
    
    Parameters
    ----------
    dt : pd.Timestamp
        日期
        
    Returns
    -------
    int
        1=节假日, 0=非节假日
    """
    # 固定日期节日
    if (dt.month, dt.day) in US_MAJOR_HOLIDAYS:
        return 1
    
    # 感恩节: 11月第4个周四
    if dt.month == 11 and dt.weekday() == 3:  # 周四
        # 计算是否是第4个周四
        first_day = pd.Timestamp(dt.year, 11, 1)
        first_thursday = first_day + pd.Timedelta(days=(3 - first_day.weekday()) % 7)
        fourth_thursday = first_thursday + pd.Timedelta(weeks=3)
        if dt.date() == fourth_thursday.date():
            return 1
    
    # 劳动节: 9月第1个周一
    if dt.month == 9 and dt.weekday() == 0:  # 周一
        if dt.day <= 7:  # 第一个周一
            return 1
    
    # 阵亡将士纪念日: 5月最后一个周一
    if dt.month == 5 and dt.weekday() == 0:  # 周一
        # 检查下周是否还在5月
        next_monday = dt + pd.Timedelta(weeks=1)
        if next_monday.month != 5:
            return 1
    
    return 0


# =============================================================================
# 第四部分: 特征工程器
# =============================================================================

class DAMFeatureEngineer:
    """
    DAM电价预测特征工程器
    
    将小时级DAM价格数据转换为预测特征数据集
    每一行代表一个预测任务: 在D-1日预测D日某小时的价格
    
    Attributes
    ----------
    config : DAMFeatureConfig
        特征提取配置
    feature_names : List[str]
        特征名列表 (不含target和timestamp)
    categorical_indices : List[int]
        分类特征的索引
    """
    
    def __init__(self, config: DAMFeatureConfig = None):
        """
        初始化特征工程器
        
        Parameters
        ----------
        config : DAMFeatureConfig, optional
            配置对象，如果为None则使用默认配置
        """
        self.config = config or DAMFeatureConfig()
        self.feature_names = ALL_FEATURES.copy()
        self.categorical_indices = CATEGORICAL_INDICES.copy()
        self.categorical_features = CATEGORICAL_FEATURES.copy()
    
    def extract_features(
        self, 
        dam_df: pd.DataFrame,
        verbose: bool = True
    ) -> pd.DataFrame:
        """
        从DAM价格数据中提取预测特征
        
        Parameters
        ----------
        dam_df : pd.DataFrame
            DAM价格数据，必须包含:
            - index: DatetimeIndex (小时级时间戳)
            - dam_price: 价格列
        verbose : bool
            是否打印进度信息
            
        Returns
        -------
        pd.DataFrame
            特征数据集，包含:
            - 35个特征列
            - target: 目标价格 (D日该小时的实际价格)
            - timestamp: 预测时间戳 (D-1日 06:00)
            
        Notes
        -----
        timestamp列说明:
        - 表示预测任务的时间戳，即D-1日06:00
        - 可用于后续按时间划分训练/测试集
        - 格式: YYYY-MM-DD HH:MM:SS
        """
        if verbose:
            print("=" * 60)
            print("DAM特征提取开始")
            print("=" * 60)
        
        # -----------------------------------------
        # 1. 数据预处理
        # -----------------------------------------
        dam_df = dam_df.copy()
        
        # 确保索引是DatetimeIndex
        if not isinstance(dam_df.index, pd.DatetimeIndex):
            raise ValueError("dam_df的索引必须是DatetimeIndex")
        
        # 添加日期和小时列
        dam_df['date'] = dam_df.index.date
        dam_df['hour'] = dam_df.index.hour + 1  # 转换为HE1-HE24
        
        # 创建日期-小时透视表 (行=日期, 列=小时1-24)
        price_pivot = dam_df.pivot_table(
            index='date',
            columns='hour',
            values='dam_price',
            aggfunc='first'
        )
        
        dates = price_pivot.index.tolist()
        n_dates = len(dates)
        
        if verbose:
            print(f"数据时间范围: {dates[0]} ~ {dates[-1]}")
            print(f"总天数: {n_dates}")
            print(f"最少历史需求: {self.config.min_history_days} 天")
        
        # -----------------------------------------
        # 2. 提取特征
        # -----------------------------------------
        records = []
        start_idx = self.config.min_history_days  # 从第29天开始 (需要28天历史)
        
        if verbose:
            print(f"\n开始提取特征...")
            print(f"有效预测天数: {n_dates - start_idx}")
        
        for i in range(start_idx, n_dates):
            target_date = dates[i]  # D日
            
            # 遍历每个小时
            for target_hour in self.config.hours:
                # 获取目标价格
                if target_hour not in price_pivot.columns:
                    continue
                    
                target_price = price_pivot.loc[target_date, target_hour]
                if pd.isna(target_price):
                    continue
                
                # 构建特征
                features = self._build_features(
                    price_pivot=price_pivot,
                    dates=dates,
                    target_date_idx=i,
                    target_hour=target_hour
                )
                
                if features is None:
                    continue
                
                # 添加目标值
                features['target'] = target_price
                
                # 添加时间戳: D-1日 06:00 (预测执行时间)
                d1_date = dates[i - 1]
                features['timestamp'] = pd.Timestamp(d1_date) + pd.Timedelta(hours=6)
                
                records.append(features)
        
        # -----------------------------------------
        # 3. 创建DataFrame
        # -----------------------------------------
        df = pd.DataFrame(records)
        
        # 重新排序列: 特征列 + target + timestamp
        col_order = self.feature_names + ['target', 'timestamp']
        df = df[col_order]
        
        if verbose:
            print(f"\n特征提取完成!")
            print(f"  总记录数: {len(df):,}")
            print(f"  特征数量: {len(self.feature_names)}")
            print(f"  分类特征数量: {len(self.categorical_indices)}")
            print(f"  时间戳范围: {df['timestamp'].min()} ~ {df['timestamp'].max()}")
        
        return df
    
    def _build_features(
        self,
        price_pivot: pd.DataFrame,
        dates: List,
        target_date_idx: int,
        target_hour: int
    ) -> Optional[Dict]:
        """
        为单个预测任务构建特征
        
        Parameters
        ----------
        price_pivot : pd.DataFrame
            价格透视表 (日期 x 小时)
        dates : List
            日期列表
        target_date_idx : int
            目标日期在列表中的索引
        target_hour : int
            目标小时 (1-24)
            
        Returns
        -------
        Dict or None
            特征字典，如果数据不足则返回None
        """
        features = {}
        
        # 关键日期
        target_date = dates[target_date_idx]      # D日 (预测目标)
        d1_date = dates[target_date_idx - 1]      # D-1日
        d2_date = dates[target_date_idx - 2]      # D-2日
        d3_date = dates[target_date_idx - 3]      # D-3日
        d7_date = dates[target_date_idx - 7]      # D-7日
        
        # 获取历史价格
        try:
            d1_prices = price_pivot.loc[d1_date].values    # D-1日24小时
            d2_prices = price_pivot.loc[d2_date].values    # D-2日24小时
            d3_prices = price_pivot.loc[d3_date].values    # D-3日24小时
            d7_prices = price_pivot.loc[d7_date].values    # D-7日24小时
        except KeyError:
            return None
        
        # 检查数据完整性
        if np.any(pd.isna(d1_prices)) or np.any(pd.isna(d7_prices)):
            return None
        
        hour_idx = target_hour - 1  # 转换为0-based索引
        
        # =================================================================
        # 目标时间特征 (10个) - 索引 0-9
        # =================================================================
        target_dt = pd.Timestamp(target_date)
        
        features['hour'] = target_hour                              # 0 [CAT]
        features['day_of_week'] = target_dt.dayofweek               # 1 [CAT]
        features['day_of_month'] = target_dt.day                    # 2 [CAT]
        features['month'] = target_dt.month                         # 3 [CAT]
        features['quarter'] = target_dt.quarter                     # 4 [CAT]
        features['week_of_year'] = target_dt.isocalendar()[1]       # 5
        features['is_weekend'] = int(target_dt.dayofweek >= 5)      # 6 [CAT]
        features['is_holiday'] = is_us_holiday(target_dt)           # 7 [CAT]
        features['is_peak_hour'] = int(target_hour in self.config.peak_hours)  # 8 [CAT]
        features['is_summer'] = int(target_dt.month in self.config.summer_months)  # 9 [CAT]
        
        # =================================================================
        # 滞后特征 (12个) - 索引 10-21
        # =================================================================
        
        # D-1日同小时 (最重要的特征!)
        features['dam_lag_24h'] = d1_prices[hour_idx]               # 10
        
        # D-1日相邻小时
        features['dam_lag_24h_prev'] = d1_prices[max(0, hour_idx - 1)]      # 11
        features['dam_lag_24h_next'] = d1_prices[min(23, hour_idx + 1)]     # 12
        
        # D-2日同小时
        features['dam_lag_48h'] = d2_prices[hour_idx]               # 13
        
        # D-7日同小时 (上周同天同时)
        features['dam_lag_168h'] = d7_prices[hour_idx]              # 14
        
        # D-1日统计
        features['dam_d1_mean'] = np.nanmean(d1_prices)             # 15
        features['dam_d1_max'] = np.nanmax(d1_prices)               # 16
        features['dam_d1_min'] = np.nanmin(d1_prices)               # 17
        features['dam_d1_std'] = np.nanstd(d1_prices)               # 18
        
        # 过去7天价格
        prices_7d = []
        for j in range(1, 8):
            if target_date_idx - j >= 0:
                day_prices = price_pivot.loc[dates[target_date_idx - j]].values
                prices_7d.extend(day_prices[~pd.isna(day_prices)])
        
        features['dam_7d_mean'] = np.mean(prices_7d) if prices_7d else features['dam_d1_mean']  # 19
        
        # 过去7天同小时均价
        same_hour_7d = []
        for j in range(1, 8):
            if target_date_idx - j >= 0:
                val = price_pivot.loc[dates[target_date_idx - j], target_hour]
                if not pd.isna(val):
                    same_hour_7d.append(val)
        features['dam_7d_same_hour_mean'] = np.mean(same_hour_7d) if same_hour_7d else features['dam_lag_24h']  # 20
        
        # 过去4周同星期同小时均价
        same_dow_hour = []
        for w in range(1, 5):
            idx = target_date_idx - 7 * w
            if idx >= 0:
                val = price_pivot.loc[dates[idx], target_hour]
                if not pd.isna(val):
                    same_dow_hour.append(val)
        features['dam_4w_same_dow_hour_mean'] = np.mean(same_dow_hour) if same_dow_hour else features['dam_lag_168h']  # 21
        
        # =================================================================
        # 模式特征 (8个) - 索引 22-29
        # =================================================================
        
        # 日内模式比例
        features['dam_d1_hour_ratio'] = features['dam_lag_24h'] / (features['dam_d1_mean'] + 1e-6)  # 22
        features['dam_7d_hour_ratio'] = features['dam_7d_same_hour_mean'] / (features['dam_7d_mean'] + 1e-6)  # 23
        
        # 趋势特征
        d2_mean = np.nanmean(d2_prices)
        d7_mean = np.nanmean(d7_prices)
        features['dam_d1_vs_d2'] = features['dam_d1_mean'] - d2_mean        # 24
        features['dam_d1_vs_d7'] = features['dam_d1_mean'] - d7_mean        # 25
        
        # 3天趋势 (简单线性斜率)
        d3_mean = np.nanmean(d3_prices)
        features['dam_trend_3d'] = (features['dam_d1_mean'] - d3_mean) / 3  # 26
        
        # 波动特征
        features['dam_7d_cv'] = np.std(prices_7d) / (np.mean(prices_7d) + 1e-6) if prices_7d else 0  # 27
        features['dam_d1_range'] = features['dam_d1_max'] - features['dam_d1_min']  # 28
        features['dam_7d_same_hour_std'] = np.std(same_hour_7d) if len(same_hour_7d) > 1 else 0  # 29
        
        # =================================================================
        # 尖峰特征 (5个) - 索引 30-34
        # =================================================================
        threshold = self.config.spike_threshold
        
        features['dam_d1_spike_count'] = int(np.sum(d1_prices > threshold))  # 30
        features['dam_7d_spike_count'] = int(np.sum(np.array(prices_7d) > threshold)) if prices_7d else 0  # 31
        features['dam_d1_had_spike'] = int(features['dam_d1_spike_count'] > 0)  # 32 [CAT]
        features['dam_lag_24h_is_spike'] = int(features['dam_lag_24h'] > threshold)  # 33 [CAT]
        features['dam_d1_max_hour'] = int(np.nanargmax(d1_prices) + 1)  # 34 (HE1-HE24)
        
        return features
    
    def get_feature_names(self) -> List[str]:
        """获取特征名列表"""
        return self.feature_names
    
    def get_categorical_indices(self) -> List[int]:
        """获取分类特征索引"""
        return self.categorical_indices
    
    def get_categorical_features(self) -> List[str]:
        """获取分类特征名列表"""
        return self.categorical_features
    
    def print_feature_info(self):
        """打印特征信息"""
        print("=" * 70)
        print("DAM预测特征列表")
        print("=" * 70)
        print(f"\n总特征数: {len(self.feature_names)}")
        print(f"分类特征数: {len(self.categorical_indices)}")
        print(f"数值特征数: {len(self.feature_names) - len(self.categorical_indices)}")
        
        print("\n" + "-" * 70)
        print(f"{'索引':<6} {'特征名':<30} {'类型':<8} {'说明'}")
        print("-" * 70)
        
        descriptions = [
            "目标小时 (HE1-HE24)",
            "目标日星期几 (0-6)",
            "目标日日期 (1-31)",
            "目标日月份 (1-12)",
            "目标日季度 (1-4)",
            "目标日年内周数",
            "目标日是否周末",
            "目标日是否节假日",
            "目标小时是否峰时",
            "目标日是否夏季",
            "D-1日同小时价格 ⭐最重要",
            "D-1日前一小时价格",
            "D-1日后一小时价格",
            "D-2日同小时价格",
            "D-7日同小时价格",
            "D-1日均价",
            "D-1日最高价",
            "D-1日最低价",
            "D-1日标准差",
            "过去7天均价",
            "过去7天同小时均价",
            "过去4周同星期同小时均价",
            "D-1日该小时/均价比",
            "7天同小时/均价比",
            "D-1日均价 - D-2日均价",
            "D-1日均价 - D-7日均价",
            "3天价格趋势",
            "7天变异系数",
            "D-1日价格范围",
            "7天同小时标准差",
            "D-1日尖峰次数",
            "7天尖峰次数",
            "D-1日是否有尖峰",
            "D-1日同小时是否尖峰",
            "D-1日最高价小时",
        ]
        
        for i, (name, desc) in enumerate(zip(self.feature_names, descriptions)):
            feat_type = "[CAT]" if i in self.categorical_indices else ""
            print(f"{i:<6} {name:<30} {feat_type:<8} {desc}")
        
        print("\n" + "-" * 70)
        print("分类特征索引:", self.categorical_indices)
        print("-" * 70)


# =============================================================================
# 第五部分: 数据生成 (用于测试)
# =============================================================================

def generate_mock_dam_data(
    start_date: str = '2015-01-01',
    end_date: str = '2025-12-31',
    seed: int = 42
) -> pd.DataFrame:
    """
    生成模拟DAM历史数据 (用于测试)
    
    Parameters
    ----------
    start_date : str
        开始日期
    end_date : str
        结束日期
    seed : int
        随机种子
        
    Returns
    -------
    pd.DataFrame
        包含 'dam_price' 列的DataFrame，index为时间戳
    """
    np.random.seed(seed)
    
    # 生成小时级时间序列
    dates = pd.date_range(start_date, end_date, freq='h')
    n = len(dates)
    
    # 基础价格
    base_price = 35
    
    # 日内模式
    hour = dates.hour
    daily_pattern = (
        15 * np.exp(-0.5 * ((hour - 9) / 3) ** 2) +
        20 * np.exp(-0.5 * ((hour - 19) / 3) ** 2) -
        10 * np.exp(-0.5 * ((hour - 4) / 2) ** 2)
    )
    
    # 季节性
    day_of_year = dates.dayofyear
    seasonal = 12 * np.sin(2 * np.pi * (day_of_year - 80) / 365)
    
    # 夏季加成
    is_summer = dates.month.isin([6, 7, 8, 9])
    summer_premium = np.where(is_summer, 8, 0)
    
    # 周末效应
    is_weekend = dates.dayofweek >= 5
    weekend_effect = np.where(is_weekend, -8, 0)
    
    # 噪声
    noise = np.random.normal(0, 4, n)
    
    # 尖峰
    spike_prob = np.where(
        is_summer & (hour >= 14) & (hour <= 20),
        0.015, 0.003
    )
    has_spike = np.random.random(n) < spike_prob
    spike_magnitude = np.random.exponential(80, n)
    spikes = has_spike * spike_magnitude
    
    # 合成价格
    price = base_price + daily_pattern + seasonal + summer_premium + weekend_effect + noise + spikes
    price = np.clip(price, -50, 2000)
    
    dam_df = pd.DataFrame({'dam_price': price}, index=dates)
    dam_df.index.name = 'timestamp'
    
    return dam_df


# =============================================================================
# 第六部分: 主程序
# =============================================================================

def main():
    """
    主程序: 演示DAM特征提取流程
    """
    print("=" * 70)
    print("ERCOT DAM 电价预测 - 特征提取")
    print("=" * 70)
    
    # -----------------------------------------
    # 1. DAM历史数据
    # -----------------------------------------
    df = pd.read_csv('./processed_data/dam_houston_price.csv')
    df['date'] = pd.to_datetime(df['date'], format='%m/%d/%Y')
    df['hour'] = pd.to_timedelta((df['hour'].astype(int) - 1).clip(upper=23), unit='h')
    df['dt'] = df['date'] + df['hour']
    df.index = df.dt
    dam_df = df.iloc[:, 2:3]
    print(f"  数据量: {len(dam_df):,} 条")
    print(f"  时间范围: {dam_df.index.min()} ~ {dam_df.index.max()}")
    
    # -----------------------------------------
    # 2. 初始化特征工程器
    # -----------------------------------------
    print("\n【步骤2】初始化特征工程器...")
    config = DAMFeatureConfig(
        spike_threshold=100.0,
        min_history_days=28
    )
    fe = DAMFeatureEngineer(config)
    
    # 打印特征信息
    fe.print_feature_info()
    
    # -----------------------------------------
    # 3. 提取特征
    # -----------------------------------------
    print("\n【步骤3】提取特征...")
    feature_df = fe.extract_features(dam_df, verbose=True)
    
    # -----------------------------------------
    # 4. 查看结果
    # -----------------------------------------
    print("\n【步骤4】数据集预览...")
    print("\n前5行:")
    print(feature_df.head())
    
    print("\n后5行:")
    print(feature_df.tail())
    
    print("\n数据类型:")
    print(feature_df.dtypes)
    
    print("\n统计摘要:")
    print(feature_df.describe())
    
    # -----------------------------------------
    # 5. 按时间戳划分数据集示例
    # -----------------------------------------
    print("\n【步骤5】按时间戳划分数据集示例...")
    
    train_end = '2024-06-30 06:00:00'
    val_end = '2024-09-30 06:00:00'
    
    train_df = feature_df[feature_df['timestamp'] <= train_end]
    val_df = feature_df[(feature_df['timestamp'] > train_end) & (feature_df['timestamp'] <= val_end)]
    test_df = feature_df[feature_df['timestamp'] > val_end]
    
    print(f"  训练集: {len(train_df):,} 条 (截止 {train_end})")
    print(f"  验证集: {len(val_df):,} 条 ({train_end} ~ {val_end})")
    print(f"  测试集: {len(test_df):,} 条 ({val_end} 之后)")
    
    # -----------------------------------------
    # 6. 保存结果
    # -----------------------------------------
    print("\n【步骤6】保存结果...")
    output_path = './train_data/dam_features.csv'
    feature_df.to_csv(output_path, index=False)
    print(f"  特征数据已保存: {output_path}")

    
    print("\n" + "=" * 70)
    print("特征提取完成!")
    print("=" * 70)
    
    return feature_df, fe

In [39]:
# =============================================================================
# 运行
# =============================================================================
if __name__ == "__main__":
    feature_df, fe = main()

ERCOT DAM 电价预测 - 特征提取
  数据量: 95,928 条
  时间范围: 2015-01-01 00:00:00 ~ 2025-12-10 23:00:00

【步骤2】初始化特征工程器...
DAM预测特征列表

总特征数: 35
分类特征数: 11
数值特征数: 24

----------------------------------------------------------------------
索引     特征名                            类型       说明
----------------------------------------------------------------------
0      hour                           [CAT]    目标小时 (HE1-HE24)
1      day_of_week                    [CAT]    目标日星期几 (0-6)
2      day_of_month                   [CAT]    目标日日期 (1-31)
3      month                          [CAT]    目标日月份 (1-12)
4      quarter                        [CAT]    目标日季度 (1-4)
5      week_of_year                            目标日年内周数
6      is_weekend                     [CAT]    目标日是否周末
7      is_holiday                     [CAT]    目标日是否节假日
8      is_peak_hour                   [CAT]    目标小时是否峰时
9      is_summer                      [CAT]    目标日是否夏季
10     dam_lag_24h                             D-1日同小时价格 ⭐最重要
11     dam_lag_24h_p

In [41]:
feature_df.columns

Index(['hour', 'day_of_week', 'day_of_month', 'month', 'quarter',
       'week_of_year', 'is_weekend', 'is_holiday', 'is_peak_hour', 'is_summer',
       'dam_lag_24h', 'dam_lag_24h_prev', 'dam_lag_24h_next', 'dam_lag_48h',
       'dam_lag_168h', 'dam_d1_mean', 'dam_d1_max', 'dam_d1_min', 'dam_d1_std',
       'dam_7d_mean', 'dam_7d_same_hour_mean', 'dam_4w_same_dow_hour_mean',
       'dam_d1_hour_ratio', 'dam_7d_hour_ratio', 'dam_d1_vs_d2',
       'dam_d1_vs_d7', 'dam_trend_3d', 'dam_7d_cv', 'dam_d1_range',
       'dam_7d_same_hour_std', 'dam_d1_spike_count', 'dam_7d_spike_count',
       'dam_d1_had_spike', 'dam_lag_24h_is_spike', 'dam_d1_max_hour', 'target',
       'timestamp'],
      dtype='object')

In [40]:
feature_df

,hour,day_of_week,day_of_month,month,quarter,week_of_year,is_weekend,is_holiday,is_peak_hour,is_summer,...,dam_7d_cv,dam_d1_range,dam_7d_same_hour_std,dam_d1_spike_count,dam_7d_spike_count,dam_d1_had_spike,dam_lag_24h_is_spike,dam_d1_max_hour,target,timestamp
0,1,3,29,1,1,5,0,0,0,0,...,0.242423,18.88,2.165468,0,0,0,0,7,17.50,2015-01-28 06:00:00
1,2,3,29,1,1,5,0,0,0,0,...,0.242423,18.88,3.776119,0,0,0,0,7,16.25,2015-01-28 06:00:00
2,3,3,29,1,1,5,0,0,0,0,...,0.242423,18.88,4.431289,0,0,0,0,7,16.91,2015-01-28 06:00:00
3,4,3,29,1,1,5,0,0,0,0,...,0.242423,18.88,4.589275,0,0,0,0,7,17.89,2015-01-28 06:00:00
4,5,3,29,1,1,5,0,0,0,0,...,0.242423,18.88,2.483659,0,0,0,0,7,18.50,2015-01-28 06:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94712,20,2,10,12,4,50,0,0,1,0,...,0.375394,50.87,9.108228,0,0,0,0,7,60.05,2025-12-09 06:00:00
94713,21,2,10,12,4,50,0,0,1,0,...,0.375394,50.87,9.517132,0,0,0,0,7,63.34,2025-12-09 06:00:00
94714,22,2,10,12,4,50,0,0,1,0,...,0.375394,50.87,9.575899,0,0,0,0,7,57.15,2025-12-09 06:00:00
94715,23,2,10,12,4,50,0,0,0,0,...,0.375394,50.87,10.034284,0,0,0,0,7,49.72,2025-12-09 06:00:00
